# M15 — Represent Meaning and Data as Vectors

**Objective:** use vectors operationally before formalizing vector algebra.

Learning loop: **label → predict → operate → measure → compare → diagnose → decide**. This CPU-only lab uses local synthetic fixtures, small precomputed teaching embeddings, and no network, paid API, or secret.

## 1. Start with the whole: rank meaning

**Predict before running:** Which document should be most directionally similar to `how-vectors-measure-similarity`? Record an order and a reason before calculating.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

search_roots = [Path.cwd(), Path.cwd().parent]
fixture_path = next(
    (root / 'datasets' / 'M15' / 'vector_fixtures.json' for root in search_roots
     if (root / 'datasets' / 'M15' / 'vector_fixtures.json').is_file()),
    None,
)
if fixture_path is None:
    raise FileNotFoundError('Run from the repository root or labs directory')

repo_root = fixture_path.parents[2]
module_dir = repo_root / 'missions' / 'M15'
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from vector_ops import (
    add, compare_vectors, cosine_similarity, dot, euclidean_distance,
    norm, normalize, rank_vectors, subtract,
)

fixtures = json.loads(fixture_path.read_text(encoding='utf-8'))
np.set_printoptions(precision=4, suppress=True)
print(f'Loaded deterministic fixtures from {fixture_path.relative_to(repo_root)}')

In [ ]:
semantic = fixtures['semantic_embeddings']
whole_first_ranking = rank_vectors(
    semantic['query']['vector'], semantic['documents'], metric='cosine'
)
observed_order = [item['id'] for item in whole_first_ranking]
print(whole_first_ranking)
assert observed_order == semantic['expected_cosine_order']

**Compare prediction with observation.** The scores came from a representation, preprocessing policy, and metric—not from meaning in the abstract. The embeddings are hand-authored teaching fixtures, not model-quality benchmarks.

## 2. A vector keeps an ordered representation

**Predict before running:** What information would be lost if the numeric values were kept but the dimension labels were reordered or removed?

In [ ]:
numeric = fixtures['numeric_features']
feature_labels = numeric['dimensions']
feature_matrix = np.asarray([row['vector'] for row in numeric['records']], dtype=float)
record_ids = [row['id'] for row in numeric['records']]
assert feature_matrix.shape == (len(record_ids), len(feature_labels))
print('dimensions:', feature_labels)
for record_id, vector in zip(record_ids, feature_matrix, strict=True):
    print(record_id, dict(zip(feature_labels, vector, strict=True)))

## 3. Addition moves a position

**Predict before running:** Starting at `(2, 1)`, where does displacement `(3, -2)` end? Sketch it or record the coordinate first.

In [ ]:
geometry = fixtures['geometry']
start = geometry['start_position']
displacement = geometry['displacement']
end = add(start, displacement)
print(dict(zip(geometry['dimensions'], end, strict=True)))
assert np.allclose(end, geometry['expected_end_position'])

**Predict before running:** If subtraction reverses the move, what should `end - start` recover?

In [ ]:
recovered_displacement = subtract(end, start)
print('recovered displacement:', recovered_displacement)
assert np.allclose(recovered_displacement, displacement)
assert np.allclose(subtract(add(start, displacement), start), displacement)

## 4. Norm measures magnitude

**Predict before running:** Is the displacement `(3, -2)` longer or shorter than 4 units? Estimate before calculating its L2 norm.

In [ ]:
displacement_magnitude = norm(displacement)
print('magnitude:', round(displacement_magnitude, 6))
assert math.isclose(displacement_magnitude, math.sqrt(13.0))

## 5. Normalization keeps direction and removes magnitude

**Predict before running:** What must the norm of a correctly L2-normalized nonzero vector be? Does this operation rescale heterogeneous feature columns across a dataset?

In [ ]:
unit_displacement = normalize(displacement)
print('unit direction:', unit_displacement)
assert math.isclose(norm(unit_displacement), 1.0)
assert np.allclose(np.asarray(unit_displacement) * displacement_magnitude, displacement)

## 6. Dot product combines alignment and magnitude

**Predict before running:** Order aligned, orthogonal, and opposed directions by dot product with `(1, 0)`. What changes if an aligned candidate becomes five times longer?

In [ ]:
reference = (1.0, 0.0)
directions = {
    'aligned': (2.0, 0.0),
    'orthogonal': (0.0, 3.0),
    'opposed': (-1.0, 0.0),
}
dot_scores = {name: dot(reference, vector) for name, vector in directions.items()}
print(dot_scores)
assert dot_scores == {'aligned': 2.0, 'orthogonal': 0.0, 'opposed': -1.0}

## 7. Cosine measures directional agreement

**Predict before running:** Should scaling `(2, 0)` to `(20, 0)` change cosine similarity with `(1, 0)`? Explain before calculating.

In [ ]:
cosine_scores = {
    name: cosine_similarity(reference, vector)
    for name, vector in directions.items()
}
print(cosine_scores)
assert math.isclose(cosine_similarity(reference, (2.0, 0.0)), 1.0)
assert math.isclose(cosine_similarity(reference, (20.0, 0.0)), 1.0)
assert cosine_scores == {'aligned': 1.0, 'orthogonal': 0.0, 'opposed': -1.0}

## 8. Euclidean distance measures absolute closeness

**Predict before running:** Which is closer to `(1, 0)`: `(1, 0.8)` or `(5, 0)`? This question is about location, not direction.

In [ ]:
nearby_distance = euclidean_distance((1.0, 0.0), (1.0, 0.8))
far_distance = euclidean_distance((1.0, 0.0), (5.0, 0.0))
print({'nearby': nearby_distance, 'same_direction_far': far_distance})
assert nearby_distance < far_distance
assert math.isclose(
    euclidean_distance((1.0, 0.8), (1.0, 0.0)), nearby_distance
)

## 9. Similarity ranking is an explicit policy

**Predict before running:** Will the semantic cosine order still match the whole-first result? Notice that `rank_vectors` requires the metric and resolves equal scores by candidate ID.

In [ ]:
semantic_ranking = rank_vectors(
    semantic['query']['vector'], semantic['documents'], metric='cosine'
)
print([(row['id'], round(float(row['score']), 4)) for row in semantic_ranking])
assert [row['id'] for row in semantic_ranking] == semantic['expected_cosine_order']

## 10. Controlled failure: cosine and Euclidean disagree

**Predict before running:** Which candidate wins when direction matters? Which wins when absolute closeness matters? A disagreement is expected and is not itself a bug.

In [ ]:
failure = fixtures['metric_disagreement']
cosine_ranking = rank_vectors(failure['query'], failure['candidates'], metric='cosine')
euclidean_ranking = rank_vectors(failure['query'], failure['candidates'], metric='euclidean')
cosine_winner = cosine_ranking[0]['id']
euclidean_winner = euclidean_ranking[0]['id']
print('cosine:', cosine_ranking)
print('euclidean:', euclidean_ranking)
assert cosine_winner == failure['expected_top']['cosine']
assert euclidean_winner == failure['expected_top']['euclidean']
assert cosine_winner != euclidean_winner

## 11. Make preprocessing part of the contract

**Predict before running:** After L2-normalizing the query and every candidate, should Euclidean ordering agree with cosine ordering? This normalization removes each vector's magnitude; it is not feature-column scaling.

In [ ]:
normalized_query = normalize(failure['query'])
normalized_candidates = [
    {'id': row['id'], 'vector': normalize(row['vector'])}
    for row in failure['candidates']
]
normalized_euclidean = rank_vectors(
    normalized_query, normalized_candidates, metric='euclidean'
)
print('normalized Euclidean:', normalized_euclidean)
assert [row['id'] for row in normalized_euclidean] == [
    row['id'] for row in cosine_ranking
]

## 12. Controlled normalization mistake

**Predict before running:** A short candidate is perfectly aligned; a longer candidate is tilted. Which wins under cosine, and which wins if a raw dot product is mislabeled as cosine?

In [ ]:
normalization_case = [
    {'id': 'short-perfect-direction', 'vector': [1.0, 0.0]},
    {'id': 'large-tilted-direction', 'vector': [4.0, 3.0]},
]
cosine_case = rank_vectors(reference, normalization_case, metric='cosine')
raw_dot_case = rank_vectors(reference, normalization_case, metric='dot')
print('cosine:', cosine_case)
print('raw dot:', raw_dot_case)
assert cosine_case[0]['id'] == 'short-perfect-direction'
assert raw_dot_case[0]['id'] == 'large-tilted-direction'

## 13. Boundary: the zero vector has no direction

**Predict before running:** Should normalization return `(0, 0)`, divide by zero, or reject the operation? State the downstream ranking risk first.

In [ ]:
try:
    normalize((0.0, 0.0))
except ValueError as error:
    print(type(error).__name__, str(error))
    assert 'zero vector' in str(error)
else:
    raise AssertionError('zero-vector normalization must fail explicitly')

## 14. Connect to V04 Mathematical Instrumentation Layer

**Predict before running:** Which measurements will reveal why the two metric winners differ? Expect norms, dot products, cosine scores and Euclidean distances—not a score without context.

In [ ]:
measurement_report = {
    row['id']: compare_vectors(failure['query'], row['vector'])
    for row in failure['candidates']
}
for candidate_id, measurements in measurement_report.items():
    print(candidate_id, {key: round(value, 4) for key, value in measurements.items()})
assert measurement_report['same-direction-large-magnitude']['cosine'] == 1.0
assert measurement_report['nearby-mixed-direction']['euclidean'] < 1.0

## Code reading

Before reopening `rank_vectors`, trace its ordering policy by hand: cosine and dot are high-to-low, Euclidean is low-to-high, and IDs resolve equal scores deterministically. Identify where invalid dimensions, non-finite values, duplicate IDs, unknown metrics and zero-vector cosine are rejected.

## No-AI Gate

Without AI-generated code, create fresh labeled vectors and predict both rankings before calculation. Construct or find one metric disagreement, handle the zero-vector boundary, then defend an explicit metric and preprocessing policy using `missions/M15/adr_prompt.md`.

## Explain and decide

Explain in plain language: (1) what each representation's dimensions mean, (2) what addition, subtraction, norm and dot product did operationally, (3) why cosine and Euclidean disagreed, (4) why raw dot is not automatically cosine, and (5) which representation + preprocessing + metric contract belongs in V04. Record remaining uncertainty and a revisit condition in the ADR.